In [2]:
# load functions
import numpy as np
import pandas as pd
import random
import os

random.seed(7)

from src.utils import get_data
from evaluation_function import *
from functools import reduce

# Performance Metrics

In [ ]:
# Check bootstrap results

#load bootstrap elastic net
#Frailty
ft_allprot_en = np.load('output_linear/bootstrap/frailty/keep_metric_frailty_allprot_own_size.npy')[:,1]
ft_cmb_en = np.load('output_linear/bootstrap/frailty/keep_metric_frailty_cmb_own_size.npy')[:,1]
ft_cmb_ffs_en = np.load('output_linear/bootstrap/frailty/keep_metric_frailty_cmb_ffs_own_size.npy')[:,1]
#Mortality
mort_allprot_en = np.load('output_linear/bootstrap/mort/metric_mort_allprot_own_size.npy')[:,1]
mort_cmb_en = np.load('output_linear/bootstrap/mort/metric_mort_cmb_own_size.npy')[:,1]
mort_cmb_ffs_en = np.load('output_linear/bootstrap/mort/metric_mort_cmb_ffs_own_size.npy')[:,1]

#load bootstrap neural nets
#Frailty
ft_allprot_fnn = pd.read_csv('intermediate_results/NN_Allprot_frailty_bootstrap.csv')['r2_val'].values

#Mortality
mort_allprot_fnn = pd.read_csv('intermediate_results/NN_Allprot_mort_bootstrap.csv')['concordance_val'].values

In [ ]:
# FNN vs EN
# allprot
# frailty
print("Comparison: FNN vs EN | allprot | frailty")
print(check_significance(ft_allprot_en, ft_allprot_fnn), "\n")

# mortality
print("Comparison: FNN vs EN | allprot | mortality")
print(check_significance(mort_allprot_en, mort_allprot_fnn), "\n")

# Check significance
# EN frailty allprot vs cmb
print("Comparison: EN | frailty | allprot vs cmb")
print(check_significance(ft_allprot_en, ft_cmb_en), "\n")

# EN frailty cmb vs cmb ffs
print("Comparison: EN | frailty | cmb vs cmb ffs")
print(check_significance(ft_cmb_en, ft_cmb_ffs_en), "\n")

# EN mortality allprot vs cmb
print("Comparison: EN | mortality | allprot vs cmb")
print(check_significance(mort_allprot_en, mort_cmb_en), "\n")

# EN mortality cmb vs cmb ffs
print("Comparison: EN | mortality | cmb vs cmb ffs")
print(check_significance(mort_cmb_en, mort_cmb_ffs_en), "\n")


In [ ]:
custom_palette = ['#88CCEE', '#44AA99', '#117733', '#332288', '#DDCC77', '#999933','#CC6677', '#882255', '#AA4499', '#DDDDDD']
custom_palette0 = sns.color_palette("tab10")
custom_palette1 = ['#332288', '#882255']
custom_palette2 = ['#117733','#332288']

densityplot(ft_allprot_en, ft_allprot_fnn, 'R2 allprot EN', 'R2 allprot FNN',custom_palette0, 'intermediate_results/EN_vs_NN_R2_dens.eps')
densityplot(mort_allprot_en, mort_allprot_fnn, 'C allprot EN', 'C allprot FNN', custom_palette0, 'intermediate_results/EN_vs_NN_C_dens.eps')

densityplot(ft_allprot_en, ft_cmb_en, 'R2 allprot EN', 'R2 CMB EN', custom_palette1, 'intermediate_results/CMB_vs_allprot_R2_dens.eps')
densityplot(mort_allprot_en, mort_cmb_en, 'C allprot EN', 'C CMB EN', custom_palette1, 'intermediate_results/CMB_vs_allprot_C_dens.eps')

densityplot(ft_cmb_en, ft_cmb_ffs_en, 'R2 CMB EN', 'R2 CMB FFS EN', custom_palette2, 'intermediate_results/FFS_vs_CMB_R2_dens.eps')
densityplot(mort_cmb_en, mort_cmb_ffs_en, 'C CMB EN', 'C CMB FFS EN', custom_palette2, 'intermediate_results/FFS_vs_CMB_C_dens.eps')

# Associations
## Newly created aging biomarkers

In [ ]:
_, set3prots, _, set3eidsall = get_data({'dset': 'cmb', 'target': 'mort', 'add_age':0, 'combine_sets':True})
_, _, _, _, _, _, colscmb = get_data({'dset': 'cmb', 'target': 'mort', 'add_age':0, 'combine_sets':False})
set3_df = pd.DataFrame(set3eidsall, columns = ['eid'], index = set3eidsall)

In [ ]:
#Elastic net biomarkers
frailty_en_directory = os.path.join('output_linear/coefs_frail/selected_models/')
mort_en_directory = os.path.join('output_linear/coefs_mort/selected_models/')
set3_df = process_en_models(frailty_en_directory, 'frailty', set3_df)
set3_df = process_en_models(mort_en_directory, 'mort', set3_df)
set3_df.index = set3_df['eid']
set3_df.drop('eid', axis =1, inplace=True)
set3_df.columns = set3_df.columns.str.removeprefix("combine_")

In [ ]:
#Neural net biomarkers
#Mortality
assign_nn_out('allprot', 'mort', "nn_models/model_pun3hzmh.pt")
assign_nn_out('cmb', 'mort', "nn_models/model_i4uflnuh.pt")
assign_nn_out("cmb_ffs", "mort", "nn_models/model_77k50cmg.pt")

#Frailty
assign_nn_out('allprot', 'frailty', "nn_models/model_m2526gyz.pt")
assign_nn_out('cmb', 'frailty', "nn_models/model_qjtdn71n.pt")
assign_nn_out("cmb_ffs", "frailty", "nn_models/model_dbwxpwou.pt")

In [ ]:
# Merge mortality and frailty datasets then combine
nn_pred_cmb_mortality = reduce(lambda left, right: pd.merge(left, right, left_index=True, right_index=True, how='outer'), 
                               [v for k, v in globals().items() if k.startswith('nn_') and k.endswith('_mort')])
nn_pred_cmb_frailty = reduce(lambda left, right: pd.merge(left, right, left_index=True, right_index=True, how='outer'), 
                             [v for k, v in globals().items() if k.startswith('nn_') and k.endswith('_frailty')])
nn_combined_data = pd.merge(nn_pred_cmb_mortality, nn_pred_cmb_frailty, left_index=True, right_index=True, how='outer')

In [ ]:
## Already published aging biomarkers

In [ ]:
gadd = pd.read_csv('Data/Other_Biomarkers/Gadd_biomarkers.csv', index_col = 'eid')
pac = pd.read_csv('Data/Other_Biomarkers/pac_set3.csv', index_col = 'eid')
metabohealth = pd.read_csv('./Data/Other_Biomarkers/Round0_MiMIR_MetaboHealth.csv', index_col = 'eid')
metabohealth = metabohealth[metabohealth.index.isin(set3_df.index)]
proteinage = pd.read_csv('./Data/Other_Biomarkers/ProteinAge.csv', index_col = 'eid')
metabohealth.dropna(how='any', inplace=True)
set3_proteins = pd.DataFrame(set3prots[0], columns = colscmb[:-1], index = set3eidsall)

## Covariate and outcome measures

In [ ]:
#Age 
age = pd.read_csv('./Data/covar/basicinfo.csv', low_memory=False, index_col = 'eid') #age
age = age[['age_center.0.0']]
age = age.rename(columns={'age_center.0.0': 'age'})

#Mortality
set3_mort = pd.read_csv("Data/Processed/Full/mort_full_val.csv", index_col = 'eid') #differently named, chosen based on number of individuals

#Frailty
frailty_set3 = pd.read_csv('Data/endpoints/Frailty/frailty_clean_set3.csv', index_col = 'eid')

#Lifestyle
lifestyle = pd.read_csv('Data/covar/UKB_Lifestyle.csv', index_col = 'eid')
lifestyle = lifestyle[['ses.0.0', 'lymf.0.0', 'mono.0.0','bmi.0.0', 'smoking.0.0', 'smokingyn.0.0', 'smkfrq.0.0', 'alcperweek.0.0']]

#CVD
cvd = pd.read_csv('Data/endpoints/FollowUpCVD.csv', index_col = 'eid')

#Cancer
cancer = pd.read_csv('Data/endpoints/FollowUpCancer.csv', index_col = 'eid')

#Handgrip and selfperceived health
#Note: originally higher score on self perceived health means lower self perceived health but changed it for easier interpretability
hgsph = pd.read_csv('Data/endpoints/HandgripSelfperceived.csv', index_col = 'eid')
hgsph['poor_health'] = np.where(hgsph['self_perceived_health'].isin([0, 1]), 1, 0)
hgsph['poor_health'] = hgsph['poor_health'].where(hgsph['self_perceived_health'].notna(), np.nan)

#Basic info
basicinfo = pd.read_csv('Data/covar/basicinfo_instance_0.csv', index_col = 'eid')

In [ ]:
#Make one giant dataframe
dfs_to_merge = [set3_df, nn_combined_data, gadd, pac, proteinage, metabohealth, set3_proteins['GDF15'], set3_mort, age, lifestyle, cvd, cancer, hgsph, basicinfo['hbp.0.0']]
compare_df = reduce(lambda left, right: pd.merge(left, right, left_index=True, right_index=True), dfs_to_merge)

# Filter dataset
compare_df_keep = compare_df.dropna(
    subset=[
        col for col in compare_df.columns
        if col.startswith(('coefs_', 'nn_', 'Gadd', 'ProteinAge'))
        or col in ['PAC', 'GDF15', 'mortScore']
    ],
    inplace=False
)

# Analyses
## Cox-PH Models

In [ ]:
#Incident outcomes
events = [['died'], ['CVD_inc'], ['cancer_inc']]
censorage = [['censorage'], ['CVD_age'], ['cancer_age']]
additional_columns = [[],
    ['sex'],
    ['sex', 'ses.0.0'],
    ['sex', 'ses.0.0', 'smokingyn.0.0', 'bmi.0.0'],
    ['sex', 'ses.0.0', 'smokingyn.0.0', 'bmi.0.0', 'lymf.0.0', 'mono.0.0']]

coefs_df = run_coxph_models(
    compare_df_keep=compare_df_keep,
    events=events,
    censorage=censorage,
    additional_columns=additional_columns,
    output_path='intermediate_results/UKBB_test_olink_scaled_biomarkers_mortality.csv',
    add_metabohealth=True,
    add_gdf15=True
)

In [ ]:
#Continuous and binary outcomes
true_outcomes = [['FI_0'], ['CVD_prev'], ['hbp.0.0'], ['cancer_prev'], ['max_handgrip'], ['poor_health']]
riskfact = [['smokingyn.0.0'], ['bmi.0.0']]
outcomes = true_outcomes + riskfact

frailty_additional_columns = [
    ['age'],
    ['age', 'sex'], 
    ['age', 'sex', 'ses.0.0'], 
    ['age', 'sex', 'ses.0.0', 'smokingyn.0.0', 'bmi.0.0'], 
    ['age', 'sex', 'ses.0.0', 'smokingyn.0.0', 'bmi.0.0', 'lymf.0.0', 'mono.0.0']
]

compare_frailty = compare_df_keep.merge(frailty_set3, left_index=True, right_index=True)

results_df = run_linlog_models(
    compare_frailty=compare_frailty,
    outcomes=outcomes,
    frailty_additional_columns=frailty_additional_columns,
    output_path='intermediate_results/UKBB_test_olink_scaled_biomarkers_frailty.csv',
    add_metabohealth=True,
    add_gdf15=True
)
